# 📏 Chapter 4: Distance Metrics and Nearest Neighbor
**Book Reference:** *scikit-learn Cookbook, Third Edition*

---
## 1. Introduction
The *Nearest Neighbor* algorithm operates on a simple principle: similar data points are located close to one another within the feature space. The core foundation of this algorithm lies in how we define and compute this "Distance" metric.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split

%matplotlib inline
np.random.seed(42)

## 2. Distance Metrics
Before building a K-NN model, we must understand how distance is calculated. The `scikit-learn` library provides `pairwise_distances` to efficiently compute distances between data points across a variety of metrics.

In [ ]:
from sklearn.metrics import pairwise_distances

# Construct two simple points in a 2D space
point_A = [[0, 0]]
point_B = [[3, 4]]

# 1. Euclidean Distance (Straight line, Pythagorean theorem)
euclidean = pairwise_distances(point_A, point_B, metric='euclidean')
print(f"Euclidean Distance (Straight line): {euclidean[0][0]}") # Expected result: 5.0

# 2. Manhattan Distance (Grid-based city block movement)
manhattan = pairwise_distances(point_A, point_B, metric='manhattan')
print(f"Manhattan Distance (3 right + 4 up): {manhattan[0][0]}") # Expected result: 7.0

## 3. K-Nearest Neighbors Classifier
We will deploy K-NN to handle classification tasks. Crucial hyperparameters to consider here are `n_neighbors` (total neighbors evaluated) and `p` (where `p=2` denotes Euclidean distance, and `p=1` denotes Manhattan distance).

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# Generate a synthetic classification dataset
X_clf, y_clf = make_classification(n_samples=300, n_features=2, n_redundant=0, n_clusters_per_class=1, random_state=42)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_clf, y_clf, test_size=0.3)

# Fit a K-NN model using 5 nearest neighbors with Euclidean distance (p=2)
knn_clf = KNeighborsClassifier(n_neighbors=5, p=2)
knn_clf.fit(X_train_c, y_train_c)

y_pred_c = knn_clf.predict(X_test_c)
print(f"KNN Classifier Accuracy: {accuracy_score(y_test_c, y_pred_c):.2f}")

# Decision Boundary Visualization Function
def plot_decision_boundary(model, X, y, title):
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.05), np.arange(y_min, y_max, 0.05))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    plt.figure(figsize=(7, 5))
    plt.contourf(xx, yy, Z, alpha=0.3, cmap='winter')
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='winter')
    plt.title(title)
    plt.show()

plot_decision_boundary(knn_clf, X_test_c, y_test_c, 'KNN Decision Boundary (K=5)')

## 4. K-Nearest Neighbors Regressor
KNN can also map continuous numerical targets. The prediction returned is the average target value of its $K$ nearest neighbors.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

# Generate 1D regression dataset for clean visualization
X_reg = np.sort(5 * np.random.rand(80, 1), axis=0)
y_reg = np.sin(X_reg).ravel() + np.random.normal(0, 0.1, 80)

knn_reg = KNeighborsRegressor(n_neighbors=5)
knn_reg.fit(X_reg, y_reg)

# Predict along a continuous line path
T = np.linspace(0, 5, 500)[:, np.newaxis]
y_pred_reg = knn_reg.predict(T)

plt.figure(figsize=(7, 5))
plt.scatter(X_reg, y_reg, color='darkorange', label='Original Data')
plt.plot(T, y_pred_reg, color='navy', linewidth=2, label='KNN Regression Prediction')
plt.title('K-Nearest Neighbors Regressor (K=5)')
plt.legend()
plt.show()

## 5. Radius Neighbors Classifier
When working with uneven data density, forcing a model to seek $K$ neighbors can introduce errors (since the 5th neighbor might sit far outside the cluster). An alternative strategy looks for any and all neighbors residing inside a fixed **radius** perimeter.

In [ ]:
from sklearn.neighbors import RadiusNeighborsClassifier

# Query all neighbors located within a 1.5 unit distance radius
radius_clf = RadiusNeighborsClassifier(radius=1.5, outlier_label='most_frequent')
radius_clf.fit(X_train_c, y_train_c)

y_pred_radius = radius_clf.predict(X_test_c)
print(f"Radius Neighbors Accuracy (Radius=1.5): {accuracy_score(y_test_c, y_pred_radius):.2f}")

plot_decision_boundary(radius_clf, X_test_c, y_test_c, 'Radius Neighbors Decision Boundary')